In [13]:
import os
import re
import sys
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from glob import glob


In [14]:
# Input files
POPULATION_PATH_UGA = "popUGA.csv"
POPULATION_PATH_COD = "popCOD.csv"

AIR_FILE_COD = "mAirCOD.csv"
AIR_FILE_UGA = "mAirUGA.csv"
LAND_FILE_COD = "mLandCOD.csv"
LAND_FILE_UGA = "mLandUGA.csv"

CASES_FILE = "cases.csv"


In [15]:
# Load population data
# For Uganda the land/air mobility files use zone NAMES (e.g. "Kampala"), while popUGA is indexed by code -> reindex popUGA by name (column "zone")
popUGA = pd.read_csv(POPULATION_PATH_UGA).set_index('zone')['pop']
popCOD = pd.read_csv(POPULATION_PATH_COD).rename({'zoneCode':'ZSCode'}, axis=1).set_index('ZSCode')['pop']
pop = pd.concat([popUGA, popCOD])


In [16]:
# Load and merge mobility data 
# Air travel matrices for COD and Uganda UGA
mAir_COD = pd.read_csv(AIR_FILE_COD).rename({'zoneCode':'ZSCode'}, axis=1)
mAir_UGA = pd.read_csv(AIR_FILE_UGA).rename({'zoneCode':'ZSCode'}, axis=1)
mAir = pd.concat([mAir_COD, mAir_UGA])

# Land travel matrices for COD and Uganda UGA
mLand_COD = pd.read_csv(LAND_FILE_COD).rename({'zoneCode':'ZSCode'}, axis=1)
mLand_UGA = pd.read_csv(LAND_FILE_UGA).rename({'zoneCode':'ZSCode'}, axis=1)
mLand = pd.concat([mLand_COD, mLand_UGA])


mLand = mLand.rename({'pax':'paxLand'}, axis=1)
mAir = mAir.rename({'pax':'paxAir'}, axis=1)

# Outer-join air and land flows on (origin zone, destination); keeping only routes wxisting in both modality

m = pd.merge(mLand, mAir, on=['ZSCode','dest'], how='outer')
m = m.fillna(0)
m['pax'] = m['paxLand'] + m['paxAir']

# Drop domestic "destinations" that are actually the source countries, and drop rows with no land traffic
m = m[~m['dest'].isin(['COD', 'UGA'])]
m = m[m['paxLand'] != 0]


In [17]:
# Load case data 

cases = pd.read_csv(CASES_FILE)


In [18]:
# Reporting rate per province/country
REPORTING_RATE = {
    'Ituri':      0.68,
    'Nord-Kivu':  0.98,
    'Sud-Kivu':   0.72,
    'Uganda':     1,
}

# HZ -> province mapping
HZ_PROVINCE = {
    # Ituri 
    'Bunia':'Ituri', 'Rwampara':'Ituri', 'Mongbalu':'Ituri', 'Nyakunde':'Ituri',
    'Bambu':'Ituri', 'Aru':'Ituri', 'Kilo':'Ituri', 'Nizi':'Ituri', 'Mangala':'Ituri',
    'Damas':'Ituri', 'Aungba':'Ituri', 'Gethy':'Ituri', 'Komanda':'Ituri', 'Lita':'Ituri',
    'Logo':'Ituri', 'Mambasa':'Ituri', 'Rimba':'Ituri', 'Tchomia':'Ituri',
    'Drodro':'Ituri', 'Nia-Nia':'Ituri', 'Fataki':'Ituri', 'Kambala':'Ituri', 'Mandima':'Ituri','Lolwa':'Ituri',
    # Nord-Kivu
    'Katwa':'Nord-Kivu', 'Beni':'Nord-Kivu', 'Butembo':'Nord-Kivu', 'Oicha':'Nord-Kivu',
    'Kalunguta':'Nord-Kivu', 'Kyondo':'Nord-Kivu', 'Goma':'Nord-Kivu',
    'Masereka':'Nord-Kivu', 'Vuhovi':'Nord-Kivu', 'Mabalako':'Nord-Kivu', 'Musienene':'Nord-Kivu',
    # Sud-Kivu
    'Miti-Murhesa':'Sud-Kivu',
    # Uganda
    'Kampala':'Uganda', 'Wakiso':'Uganda',
}


In [ ]:
# Reporting rate: two scenarios 
cases['province'] = cases['HZ'].map(HZ_PROVINCE)

cases['reporting_rate'] = cases['province'].map(REPORTING_RATE).fillna(1.0)

# OBSERVED: reporting rate = 1 for everyone
cases['cases_observed']  = cases['cases'].astype(float)
# ESTIMATED: heterogeneous reporting rate 
cases['cases_estimated'] = cases['cases'] / cases['reporting_rate']


In [ ]:
def compute_risk(cases):
    """
    Compute the importation-risk table for every destination, given a
    Series of case counts indexed by ZSCode (origin health zone).

    Method (per origin zone z, per destination d):
      1. e_z        = cases_z / pop_z                      -> incidence in zone z
      2. n_z^{mode} = total outgoing passengers of z       (summed over all destinations)
      3. alpha_z^{mode} = (n_z^{mode} * e_z) / sum_z(n_z^{mode} * e_z) -> zone z's share of total "risk-weighted mobility"
      4. r_{z,d}^{mode} = m_{z,d}^{mode} / n_z^{mode}       -> share of z's outgoing traffic that goes to d
      5. R_d^{mode} = sum_z ( r_{z,d}^{mode} * alpha_z^{mode} ) -> total risk score for destination d

    This is computed separately for air travel, land travel, and their sum.
    """
    cases = cases.astype(float)
    e = cases / pop.loc[cases.index]

    nAir  = m.groupby('ZSCode')['paxAir'].sum().reindex(cases.index)
    nLand = m.groupby('ZSCode')['paxLand'].sum().reindex(cases.index)
    n     = m.groupby('ZSCode')['pax'].sum().reindex(cases.index)

    alphaAir  = (nAir  * e) / (nAir  * e).sum()
    alphaLand = (nLand * e) / (nLand * e).sum()
    alpha     = (n     * e) / (n     * e).sum()

    # Risk per origin zone towards each destination.
    # Zero denominators (a zone with no traffic of that modality) become NaN
    # and are dropped via dropna(), instead of raising a division-by-zero.
    rAir  = m.set_index(['ZSCode','dest'])['paxAir'] / nAir.replace(0, np.nan)
    rLand = m.set_index(['ZSCode','dest'])['paxLand'] / nLand.replace(0, np.nan)
    r     = m.set_index(['ZSCode','dest'])['pax'] / n.replace(0, np.nan)

    rTotAir  = (rAir  * alphaAir ).dropna().groupby('dest').sum()
    rTotLand = (rLand * alphaLand).dropna().groupby('dest').sum()
    rTot     = (r     * alpha    ).dropna().groupby('dest').sum()

    risk = pd.concat([rTotAir, rTotLand, rTot], axis=1)
    risk.columns = ['rTotAir', 'rTotLand', 'rTot']
    return risk.sort_values('rTot')


In [ ]:
#OBSERVED (reporting rate = 1 for everyone)
risk_observed = compute_risk(cases.set_index('ZSCode')['cases_observed'])
# .sort_index() sorts alphabetically by 'dest' (the index) in the CSV
risk_observed.sort_index().to_csv('riskTot_observedCases.csv')

#ESTIMATED (heterogeneous reporting rate)
risk_estimated = compute_risk(cases.set_index('ZSCode')['cases_estimated'])
risk_estimated.sort_index().to_csv('riskTot_estimatedCases.csv')
